# 1. Locks (asyncio.Lock)
In a multi-threaded program, a race condition occurs when two threads try to modify a shared variable at the same time, leading to corrupted state. While asyncio is single-threaded, cooperative multitasking still suffers from race conditions if a coroutine yields control (await) in the middle of a critical data modification step.

* How it works: An asyncio.Lock ensures that only one coroutine can enter a specific block of code at a time. Other coroutines must wait until the lock is released.

* Syntax: You acquire the lock using async with lock: which ensures it releases automatically even if an exception occurs.

# 2. Semaphores (asyncio.Semaphore)
While a Lock restricts access to one task, a Semaphore restricts access to a maximum of $N$ tasks simultaneously.
* When to use: Rate-limiting. If you are scraping 10,000 URLs or querying a third-party payment gateway that will block your IP if you make more than 5 concurrent requests, you wrap your network call in a Semaphore.
* Syntax: Just like a lock, you use async with semaphore: to acquire and automatically release a slot.

# 3. Queues (asyncio.Queue)
An asyncio.Queue is a thread-safe (or more accurately, loop-safe) FIFO data structure designed specifically for passing messages between producer and consumer coroutines.

* When to use: Background worker patterns. For example, an API endpoint receives an image upload, drops the file path into a queue, and immediately responds to the user. A background consumer coroutine pulls items from the queue and processes them asynchronously (resizing the image, uploading to S3) without making the user wait.

In [2]:
import asyncio
import random
import time

# ==========================================
# 1. Semaphore: Rate-Limiting Concurrency
# ==========================================
async def download_file(file_id, semaphore):
    # Only allow 2 downloads to happen concurrently system-wide
    async with semaphore:
        print(f"[Download] Starting file {file_id}...")
        await asyncio.sleep(1.5)  # Simulating network download time
        print(f"[Download] Finished file {file_id}!")
        return f"File_{file_id}.pdf"

# ==========================================
# 2. Lock: Preventing Race Conditions
# ==========================================
shared_counter = 0

async def safe_increment(lock):
    global shared_counter
    async with lock:
        # Critical section: Read, modify, write
        current = shared_counter
        await asyncio.sleep(0.1) # Simulating a tiny async delay that would trigger a race condition without a lock
        shared_counter = current + 1

# ==========================================
# 3. Queue: Producer-Consumer Pipeline
# ==========================================
async def producer(queue, name):
    for i in range(3):
        item = f"Job-{name}-{i}"
        print(f"[Producer {name}] Pushing {item} to queue...")
        await queue.put(item) # Adds item to queue, yields control if queue is full
        await asyncio.sleep(0.5)

async def consumer(queue, consumer_id):
    while True:
        # Pull item from queue (waits if queue is empty)
        item = await queue.get()
        print(f"[Consumer {consumer_id}] Processing {item}...")
        await asyncio.sleep(1) # Simulating processing work
        queue.task_done() # Signal that processing for this item is complete

# ==========================================
# The Orchestrator
# ==========================================
async def main():
    start_time = time.time()
    
    # --- Demo 1: Semaphore ---
    print("--- 1. Semaphore Demo (Max 2 concurrent) ---")
    sem = asyncio.Semaphore(2)
    download_tasks = [download_file(i, sem) for i in range(1, 6)]
    await asyncio.gather(*download_tasks)
    
    # --- Demo 2: Lock ---
    print("\n--- 2. Lock Demo (Race Condition Prevention) ---")
    lock = asyncio.Lock()
    # Fire off 10 coroutines trying to increment the counter at the exact same time
    increment_tasks = [safe_increment(lock) for _ in range(10)]
    await asyncio.gather(*increment_tasks)
    print(f"Final Expected Counter Value: 10 | Actual Value: {shared_counter}")
    
    # --- Demo 3: Queue (Producer-Consumer) ---
    print("\n--- 3. Queue Pipeline Demo ---")
    queue = asyncio.Queue(maxsize=5) # Max 5 items allowed in queue buffer
    
    # Create producers and consumers
    prod1 = asyncio.create_task(producer(queue, "A"))
    prod2 = asyncio.create_task(producer(queue, "B"))
    
    # Start two workers consuming from the queue
    worker1 = asyncio.create_task(consumer(queue, 1))
    worker2 = asyncio.create_task(consumer(queue, 2))
    
    # Wait for producers to finish generating items
    await asyncio.gather(prod1, prod2)
    
    # Wait until all items in the queue have been processed and acknowledged via task_done()
    await queue.join()
    
    # Cancel the background consumers since our pipeline work is complete
    worker1.cancel()
    worker2.cancel()

    print(f"\nAll operations complete in {time.time() - start_time:.2f}s")

if __name__ == "__main__":
    #asyncio.run(main())
    await main()

--- 1. Semaphore Demo (Max 2 concurrent) ---
[Download] Starting file 1...
[Download] Starting file 2...
[Download] Finished file 1!
[Download] Finished file 2!
[Download] Starting file 3...
[Download] Starting file 4...
[Download] Finished file 3!
[Download] Finished file 4!
[Download] Starting file 5...
[Download] Finished file 5!

--- 2. Lock Demo (Race Condition Prevention) ---
Final Expected Counter Value: 10 | Actual Value: 10

--- 3. Queue Pipeline Demo ---
[Producer A] Pushing Job-A-0 to queue...
[Producer B] Pushing Job-B-0 to queue...
[Consumer 1] Processing Job-A-0...
[Consumer 2] Processing Job-B-0...
[Producer A] Pushing Job-A-1 to queue...
[Producer B] Pushing Job-B-1 to queue...
[Consumer 1] Processing Job-A-1...
[Producer A] Pushing Job-A-2 to queue...
[Producer B] Pushing Job-B-2 to queue...
[Consumer 2] Processing Job-B-1...
[Consumer 1] Processing Job-A-2...
[Consumer 2] Processing Job-B-2...

All operations complete in 8.61s
